# Caching And Persistence
------------------------------
Spark memory pool refers to:
How Spark manages and divides memory inside executors while processing data.
Spark uses memory very carefully because:
big datasets
joins
shuffles
caching
streaming
all consume large amounts of RAM.

Say an executor has 16GB of RAM , spark will divide this into different pools 
Executor Memory
   ├── Execution Memory
   └── Storage Memory

1. Execution Memory

Used for:

joins
shuffles
aggregations
sorting
computations

i.e. df.groupBy("team").count()

Spark needs memory for:

shuffle buffers
hash maps
sorting

2. Storage Memory

Used for:

caching
persistence
broadcast variables

i.e. df.cache() 
spark stores cached partitions in STORAGE memory.
This uses EXECUTION memory.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
epl_df = spark.read.table('sparkoptimization.tables.epl_final')
stadium_df = spark.read.table('sparkoptimization.tables.epl_stadiums')

# DISK_ONLY and MEMORY_ONLY
from pyspark import StorageLevel
df.persist(StorageLevel.MEMORY_ONLY)
df.persist(StorageLevel.DISK_ONLY)

| Storage Level     | Where Data Is Stored        | Pros                                                                                                   | Cons                                                                                                     | Best Use Cases                                                                          |
| ----------------- | --------------------------- | ------------------------------------------------------------------------------------------------------ | -------------------------------------------------------------------------------------------------------- | --------------------------------------------------------------------------------------- |
| `MEMORY_ONLY`     | RAM only                    | 🚀 Very fast access<br>⚡ Best performance for repeated queries<br>✅ No disk I/O                        | ❌ Can run out of memory<br>❌ Evicted partitions are recomputed<br>❌ Risk of OOM errors on large datasets | Small-medium datasets, ML workloads, iterative processing, frequently reused DataFrames |
| `DISK_ONLY`       | Local disk only             | ✅ Saves executor memory<br>✅ Can handle very large datasets<br>✅ Avoids recomputation after caching    | 🐢 Much slower than memory<br>❌ Disk reads increase latency<br>❌ More I/O overhead                       | Large ETL jobs, datasets too large for RAM, long-running transformations                |
| `MEMORY_AND_DISK` | RAM first, overflow to disk | ✅ Balanced approach<br>✅ Faster than disk-only<br>✅ Safer than memory-only<br>✅ Prevents recomputation | ❌ Some disk I/O still occurs<br>❌ Slightly more overhead managing both storage types                     | ⭐ Most common production choice for Spark ETL pipelines                                 |


In [0]:

''' By Default spark will use these filters
 (sparkoptimization.tables.epl_final.HomeFouls IS NOT NULL)
(sparkoptimization.tables.epl_final.HomeFouls >= 4L)
LIke this HomeFouls IS NOT NULL AND HomeFouls >= 4
'''

epl_df2 = epl_df.filter(col('HomeFouls') >= 4).display()


In [0]:
epl_df3 = epl_df.filter(col('FullTimeHomeGoals') >= 2).display(epl_df3)

In [0]:
### On serverless won't work.
'''1. No long-lived executors
Executors are created and destroyed per query
There is no persistent cluster memory state
Each query is treated like :
Run → Execute → Tear down resources

Automatic optimisation replaces caching ,Serverless relies on:

Photon engine
query result caching
adaptive execution (AQE)
Delta caching (storage-level)
'''

# epl_df = spark.read.table('sparkoptimization.tables.epl_final').cache()